# Runnables + Memory — Applied to Resume Checker

This notebook explains:
1. **Runnables** — the building blocks of LangChain chains
2. **Memory** — how to give your chain/agent a conversation history

All examples are tied to the resume checker use case.

---

## What is a Runnable?

Every component in LangChain (prompts, LLMs, parsers, tools) is a **Runnable**.
A Runnable has a standard interface:
- `.invoke(input)` — run once
- `.stream(input)` — stream tokens
- `.batch([input1, input2])` — run multiple at once

You chain them together with `|` (pipe operator):
```
prompt | llm | parser
```
Output of each step becomes input to the next.

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader, WebBaseLoader

llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=os.getenv('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1",
    max_tokens=2000
)
parser = StrOutputParser()

C:\Users\Mamatha-Win10\AppData\Local\Temp\ipykernel_9388\1527578169.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader, WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


---
## Step 1 — RunnableLambda

`RunnableLambda` wraps **any Python function** into a Runnable so it can be used inside a chain.

**Why use it?**  
When you need to do custom logic (e.g., load a file, clean text) in the middle of a chain.

In [2]:
# Plain function — loads a document and returns its text
def load_document(source: str) -> str:
    source_lower = source.lower()
    if source_lower.startswith(("http://", "https://")):
        loader = WebBaseLoader(source)
    elif source_lower.endswith(".pdf"):
        loader = PyPDFLoader(source)
    elif source_lower.endswith(".docx"):
        loader = Docx2txtLoader(source)
    else:
        raise ValueError(f"Unsupported format: {source}")
    docs = loader.load()
    return "\n\n".join(d.page_content for d in docs)

# Wrap it as a Runnable
load_doc_runnable = RunnableLambda(load_document)

# Now it can sit inside a chain
# Example: load file → send to LLM → parse
quick_chain = load_doc_runnable | (lambda text: f"Summarize this resume:\n{text}") | llm | parser

# quick_chain.invoke("Mamatha Resume word am.docx")

---
## Step 2 — RunnablePassthrough

`RunnablePassthrough` passes the input **unchanged** to the next step.  
Very useful when you want to carry the original input alongside transformed data.

**Why use it?**  
Your prompt needs both the raw resume text AND the job description.  
You load the resume, but you still need to pass the job description through.

In [3]:
# Scenario: input is {"source": "resume.docx", "job_description": "Data Analyst"}
# We want to load the file AND keep the job_description intact

resume_prompt = PromptTemplate(
    template="""You are a resume reviewer.
Job: {job_description}
Resume: {resume_text}
Give a match score and 3 improvement tips.""",
    input_variables=["resume_text", "job_description"]
)

# Chain that:
# 1. Loads resume text from the source path
# 2. Passes job_description through unchanged
# 3. Feeds both into the prompt

passthrough_chain = RunnableParallel({
    "resume_text": RunnableLambda(lambda x: load_document(x["source"])),
    "job_description": RunnablePassthrough() | RunnableLambda(lambda x: x["job_description"])
}) | resume_prompt | llm | parser

# Usage:
# result = passthrough_chain.invoke({
#     "source": "Mamatha Resume word am.docx",
#     "job_description": "Data Analyst"
# })
# print(result)

---
## Step 3 — RunnableParallel

`RunnableParallel` runs **multiple runnables at the same time** on the same input and merges results into a dict.

**Why use it here?**  
Run both a job-match review AND a standalone review on the same resume simultaneously — then combine both outputs.

In [ ]:
STANDALONE_PROMPT = PromptTemplate.from_template("""
Score resume standalone (clarity, format, ATS, skills):
1. **Score**: X/100
2. **Strengths**: ...
3. **Weaknesses**: ...
4. **Skills Mentioned**: ...
5. **Recommended Skills**: ...
Resume: {resume_text}
""")

JOB_MATCH_PROMPT = PromptTemplate.from_template("""
Match this resume to the job:
Job: {job_description}
Resume: {resume_text}
Give: Match %, missing keywords, top 3 suggestions.
""")

# Both chains share the same resume_text + job_description input
dual_review_chain = RunnableParallel({
    "standalone_review": STANDALONE_PROMPT | llm | parser,
    "job_match_review": JOB_MATCH_PROMPT | llm | parser
})

# Usage — both reviews run in parallel:
# result = dual_review_chain.invoke({
#     "resume_text": load_document("Mamatha Resume word am.docx"),
#     "job_description": "Data Analyst"
# })
# print("=== Standalone ===")
# print(result["standalone_review"])
# print("\n=== Job Match ===")
# print(resu lt["job_match_review"])

---
## Step 4 — Memory Concept

Without memory, every call to your chain is **stateless** — it forgets the previous conversation.  
With memory, the chain remembers past messages (like a chat).

### Two approaches:
| Approach | Use when |
|---|---|
| `RunnableWithMessageHistory` | Adding memory to a **chain** (prompt → llm → parser) |
| `MemorySaver` checkpointer | Adding memory to an **agent** (create_agent) |

We'll cover both.

---
## Step 5 — Memory for a Chain (RunnableWithMessageHistory)

This wraps your chain and automatically:
- Loads previous messages for the session
- Appends new messages after each call
- Each user session gets its own history via `session_id`

In [5]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# Store chat history per session
session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# Build a chat-style prompt that includes message history
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful resume reviewer assistant."),
    MessagesPlaceholder(variable_name="history"),  # <-- history injected here
    ("human", "{input}")
])

# Base chain (no memory yet)
base_chain = chat_prompt | llm | parser

# Wrap with memory
chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,          # function that returns history for a session
    input_messages_key="input",   # which key in input holds the user message
    history_messages_key="history" # which key in prompt holds the history
)

print("Chain with memory created!")

Chain with memory created!


d:\GENAI\Langchain_Course\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
# Each call passes a session_id — same id = same conversation history
config = {"configurable": {"session_id": "user_yamuna"}}

# Turn 1
r1 = chain_with_memory.invoke({"input": "My name is Yamuna and I am a data analyst."}, config=config)
print("Turn 1:", r1)

# Turn 2 — chain remembers Turn 1
r2 = chain_with_memory.invoke({"input": "What job roles suit me?"}, config=config)
print("\nTurn 2:", r2)

Turn 1: Nice to meet you, Yamuna! I’m excited to help you sharpen your résumé and showcase your data‑analysis expertise.

To give you the most useful feedback, could you share a bit more information? For example:

1. **Current résumé** (copy‑paste the text or a PDF link) – or the specific sections you’d like reviewed (summary, experience, projects, skills, etc.).
2. **Target role or industry** – are you aiming for a data‑analyst position at a tech startup, a senior analytics role in finance, a business‑intelligence position, etc.?
3. **Key achievements** you’d like to highlight – any measurable impacts (e.g., “improved forecasting accuracy by 15%” or “saved $200 K through process automation”).
4. **Any constraints** – page limit, ATS requirements, particular formatting preferences, etc.

Feel free to paste the text here or describe the parts you want me to focus on, and I’ll provide concrete suggestions on wording, structure, metrics, and keywords to help your résumé stand out.

Turn 2

---
## Step 6 — Apply Memory to Resume Checker Chain

Now we apply the same pattern to the resume checker.  
The user can ask follow-up questions about their resume across multiple turns.

In [7]:
# Load resume once
resume_text = load_document("Mamatha Resume word am.docx")

# Resume chat prompt — resume text is baked into the system prompt
resume_chat_prompt = ChatPromptTemplate.from_messages([
    ("system", f"""You are an expert resume reviewer.
The user's resume is below. Answer all questions based on it.

RESUME:
{resume_text}
"""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

resume_base_chain = resume_chat_prompt | llm | parser

resume_chain_with_memory = RunnableWithMessageHistory(
    resume_base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

print("Resume chain with memory ready!")

Resume chain with memory ready!


d:\GENAI\Langchain_Course\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
config = {"configurable": {"session_id": "resume_session_1"}}

# Turn 1 — initial review
r1 = resume_chain_with_memory.invoke(
    {"input": "Give me an overall score and top 3 weaknesses."},
    config=config
)
print("Turn 1:\n", r1)

# Turn 2 — follow-up (chain remembers the score from Turn 1)
r2 = resume_chain_with_memory.invoke(
    {"input": "How can I fix the first weakness you mentioned?"},
    config=config
)
print("\nTurn 2:\n", r2)

Turn 1:
 **Overall Score: 5 / 10**

Your resume shows a solid work history and clear personal details, but it needs stronger focus, clearer presentation, and more evidence of impact to stand out for a computer‑science‑related role.

---

### Top 3 Weaknesses  

| # | Weakness | Why it matters & how to improve |
|---|----------|---------------------------------|
| **1** | **Unfocused career narrative** – The objective and the listed work experience are centered on administration/HR/admissions rather than computer‑science or IT. | Recruiters can’t quickly see how your background ties to a tech role. Rewrite the **objective** (and optionally a “Profile” section) to highlight transferable skills (e.g., data handling, process automation, software usage) and state the type of tech position you’re targeting. |
| **2** | **Lack of quantifiable achievements** – Job descriptions are generic bullet lists that describe duties but give no numbers, outcomes, or impact. | Add metrics (e.g., “Processe

---
## Step 7 — Memory for Agent (MemorySaver checkpointer)

For agents created with `create_agent`, memory works differently.  
You pass a `checkpointer` — it saves the full agent state (messages + tool calls) between turns.

Each conversation thread is identified by `thread_id` in the config.

In [9]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver

def run_chain(prompt, **kwargs):
    return (prompt | llm | parser).invoke(kwargs)

@tool
def analyze_resume(source: str) -> str:
    """Analyze a resume against a job description. Pass a file path (.pdf/.docx) or URL."""
    text = load_document(source)
    return run_chain(JOB_MATCH_PROMPT, resume_text=text, job_description="front desk executive")

@tool
def resume_checker(source: str) -> str:
    """General resume review without a job description. Pass a file path (.pdf/.docx) or URL."""
    text = load_document(source)
    return run_chain(STANDALONE_PROMPT, resume_text=text)

# MemorySaver stores conversation state in memory (use SqliteSaver for persistence across restarts)
checkpointer = MemorySaver()

agent_with_memory = create_agent(
    model=llm,
    tools=[analyze_resume, resume_checker],
    system_prompt="You are a helpful resume reviewer. Remember previous questions in this conversation.",
    checkpointer=checkpointer   # <-- this enables memory
)

print("Agent with memory created!")

Agent with memory created!


In [10]:
# thread_id identifies the conversation — same id = agent remembers previous turns
config = {"configurable": {"thread_id": "thread_yamuna_1"}}

# Turn 1
r1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "Check my resume at: Mamatha Resume word am.docx"}]},
    config=config
)
print("Turn 1:\n", r1["messages"][-1].content)

# Turn 2 — agent remembers the resume was already reviewed
r2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What skills should I add based on your review?"}]},
    config=config
)
print("\nTurn 2:\n", r2["messages"][-1].content)

Turn 1:
 **Resume Review – “Mamatha Resume word am.docx”**  
*(Score out of 100 – higher = more ATS‑friendly, clearer, and more compelling)*  

| Criterion | Score | Quick Take |
|-----------|------|------------|
| **Clarity & Readability** | **12 / 25** | Information is present but cramped; headings and bullet styles are inconsistent, making the document hard to scan quickly. |
| **Formatting & Presentation** | **9 / 25** | Uses non‑standard symbols (“#”), mixed fonts/sizes, and a layout that looks more like a word‑processor draft than a polished resume. |
| **ATS Compatibility** | **10 / 25** | Lacks the standard section headings that most parsers look for; tables/odd characters can cause data loss. |
| **Skills Coverage** | **14 / 25** | Some hard‑skill keywords are there, but they are generic and not tied to measurable outcomes. |
| **Overall** | **45 / 100** | **Below‑average** – the content is solid, but the presentation and keyword‑strategic framing need major work to get past m

---
## Summary

| Concept | What it does | When to use |
|---|---|---|
| `RunnableLambda` | Wraps any Python function into a chain step | Custom logic (file loading, text cleaning) |
| `RunnablePassthrough` | Passes input unchanged to next step | Carry original data alongside transformed data |
| `RunnableParallel` | Runs multiple chains in parallel on same input | Get two reviews at once, compare outputs |
| `RunnableWithMessageHistory` | Adds memory to a **chain** | Chat-style Q&A over a resume |
| `MemorySaver` checkpointer | Adds memory to an **agent** | Multi-turn agent conversations |

**Key rule:** Use `RunnableWithMessageHistory` for chains, use `checkpointer` for agents.